# KaakiScan : Entraînement en ligne du modèle d'IA pour les maladies du Bananier

Ce notebook permet d'entraîner un réseau de neurones convolutionnel (**MobileNetV2**) pour classifier les images de bananiers en **8 classes** représentant 4 organes (Racine, Tige, Feuille, Fruit) sains ou malades :
1. `racine_saine`
2. `racine_malade` (ex: flétrissement/pourriture)
3. `tige_saine`
4. `tige_malade` (ex: charançon, flétrissement bactérien)
5. `feuille_saine`
6. `feuille_malade` (ex: Sigatoka noire)
7. `fruit_sain`
8. `fruit_malade` (ex: anthracnose)

**Comment utiliser ce notebook dans Google Colab :**
1. Ouvrez [Google Colab](https://colab.research.google.com/).
2. Cliquez sur l'onglet **Importer** (Upload) et sélectionnez ce fichier `.ipynb`.
3. Dans Colab, allez dans *Exécution > Modifier le type d'exécution* et choisissez **GPU T4** pour accélérer l'entraînement.
4. Préparez votre Dataset sous forme de fichier zip (voir ci-dessous) et lancez les cellules.

## 1. Imports et Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import os
import time
import copy
from PIL import Image

print(f"PyTorch version: {torch.__version__}")
print(f"GPU disponible : {torch.cuda.is_available()}")

## 2. Préparation du Dataset
Votre archive de dataset (ex: `dataset_banane.zip`) doit être organisée ainsi :
```text
dataset_banane/
  ├── train/
  │    ├── racine_saine/
  │    ├── racine_malade/
  │    ├── tige_saine/
  │    ├── tige_malade/
  │    ├── feuille_saine/
  │    ├── feuille_malade/
  │    ├── fruit_sain/
  │    └── fruit_malade/
  └── val/
       ├── (mêmes dossiers pour la validation/test)
```

In [ ]:
# Dézipper le dataset s'il a été téléversé sur Colab ou Drive
# Remplacez 'dataset_banane.zip' par le nom de votre fichier téléversé
if os.path.exists('dataset_banane.zip'):
    print("Décompression du dataset...")
    !unzip -q dataset_banane.zip
    print("Décompression terminée !")
else:
    print("AVERTISSEMENT : Veuillez téléverser votre fichier 'dataset_banane.zip' dans le volet gauche de Colab.")

## 3. Transformations des Données et Dataloaders

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = 'dataset_banane' # Dossier contenant 'train' et 'val'

if os.path.exists(data_dir):
    image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                      for x in ['train', 'val']}
    dataloaders = {x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=2)
                   for x in ['train', 'val']}
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
    class_names = image_datasets['train'].classes
    
    print(f"Classes détectées : {class_names}")
    print(f"Taille de l'ensemble d'entraînement : {dataset_sizes['train']} images")
    print(f"Taille de l'ensemble de validation : {dataset_sizes['val']} images")
else:
    print("Dossier dataset introuvable. Veuillez d'abord décompresser votre archive.")

## 4. Initialisation du modèle MobileNetV2 (Transfer Learning)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Charger MobileNetV2 pré-entraîné sur ImageNet
model_ft = models.mobilenet_v2(pretrained=True)

# Geler les premières couches pour ne pas écraser les poids pré-entraînés
for param in model_ft.parameters():
    param.requires_grad = False

# Remplacer la tête de classification finale
num_ftrs = model_ft.classifier[1].in_features
model_ft.classifier[1] = nn.Linear(num_ftrs, 8) # 8 classes

model_ft = model_ft.to(device)

criterion = nn.CrossEntropyLoss()

# Seuls les paramètres de la tête finale seront entraînés
optimizer_conv = optim.Adam(model_ft.classifier[1].parameters(), lr=0.001)

print(model_ft.classifier)

## 5. Fonction d'Entraînement

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    train_loss, val_loss = [], []
    train_acc, val_acc = [], []

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Mode entraînement
            else:
                model.eval()   # Mode évaluation

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'train':
                train_loss.append(epoch_loss)
                train_acc.append(epoch_acc.cpu().item())
            else:
                val_loss.append(epoch_loss)
                val_acc.append(epoch_acc.cpu().item())

            # Sauvegarder les meilleurs poids
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Entraînement complété en {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Meilleure précision en Validation : {best_acc:4f}')

    # Charger les meilleurs poids
    model.load_state_dict(best_model_wts)
    return model, train_loss, val_loss, train_acc, val_acc

## 6. Lancement de l'Entraînement

In [ ]:
# Entraîner sur 10 époques (vous pouvez augmenter à 15 ou 20)
model_ft, t_loss, v_loss, t_acc, v_acc = train_model(model_ft, criterion, optimizer_conv, num_epochs=10)

## 7. Affichage des Courbes d'Entraînement

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(t_loss, label='Train Loss')
plt.plot(v_loss, label='Val Loss')
plt.legend()
plt.title('Loss par Époque')

plt.subplot(1, 2, 2)
plt.plot(t_acc, label='Train Accuracy')
plt.plot(v_acc, label='Val Accuracy')
plt.legend()
plt.title('Accuracy par Époque')
plt.show()

## 8. Exportation du modèle entraîné
Téléchargez le fichier `banana_model.pth` généré à l'issue de cette cellule, et collez-le dans le dossier `backend/` de votre serveur FastAPI.

In [ ]:
torch.save(model_ft.state_dict(), 'banana_model.pth')
print("Modèle sauvegardé avec succès sous 'banana_model.pth' ! Cliquez sur l'icône de dossier à gauche pour le télécharger.")